# Cahoy 2010 climate-level QC (16 climate groups)

This notebook runs the **standard Aurora QC checks** at the climate-group level for `aurora_cahoy2010_replication_v0`.

Instead of checking all phase rows, it selects **one representative NetCDF per `climate_group_index`** (prefer phase 0 deg when available), then:

1. runs `validate_dataset(...)` from `aurora_grid.qc.report`,
2. creates the standard diagnostic and spectrum plots via `aurora_grid.qc.plots`,
3. builds a compact summary table and overview quality graphs.

This is a faster QC pass when you only want to verify the ~16 unique climates.

```bash
source env/activate_aurora_picaso4_job.sh
```

In [1]:
import importlib
import importlib.util
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in [cwd, *cwd.parents]
     if (p / "roadrunner_egp" / "aurora_subneptune_grid" / "src" / "aurora_grid").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root containing roadrunner_egp/aurora_subneptune_grid/src/aurora_grid")


def ensure_netcdf_backend() -> str:
    """Ensure xarray can open NetCDF4/HDF5 files on minimal HPC images."""
    if importlib.util.find_spec("h5netcdf") is not None or importlib.util.find_spec("netCDF4") is not None:
        return "system"

    target = Path("/tmp/aurora_pydeps_h5netcdf")
    if str(target) not in sys.path:
        sys.path.insert(0, str(target))

    importlib.invalidate_caches()
    if importlib.util.find_spec("h5netcdf") is not None:
        return str(target)

    target.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--target",
            str(target),
            "--no-deps",
            "h5netcdf",
        ]
    )
    importlib.invalidate_caches()
    if importlib.util.find_spec("h5netcdf") is None:
        raise RuntimeError("h5netcdf install failed; cannot open NetCDF4/HDF5 outputs")
    return str(target)


NETCDF_BACKEND_SOURCE = ensure_netcdf_backend()
import xarray as xr

GRID_ROOT = REPO_ROOT / "roadrunner_egp" / "aurora_subneptune_grid"
MODEL = "aurora_cahoy2010_replication_v0"
MANIFEST = GRID_ROOT / "manifests" / f"{MODEL}_manifest.csv"
NC_ROOT = GRID_ROOT / "outputs" / MODEL / "nc"
QC_ROOT = GRID_ROOT / "outputs" / MODEL / "qc_climate"
QC_PLOTS = QC_ROOT / "plots"
QC_ROOT.mkdir(parents=True, exist_ok=True)
QC_PLOTS.mkdir(parents=True, exist_ok=True)

for extra in (GRID_ROOT / "src", REPO_ROOT / "roadrunner_egp"):
    s = str(extra)
    if s not in sys.path:
        sys.path.insert(0, s)

from aurora_grid.qc.plots import make_qc_plot, make_spectrum_plot
from aurora_grid.qc.report import result_to_row, validate_dataset

print("MODEL:", MODEL)
print("MANIFEST:", MANIFEST)
print("NC_ROOT:", NC_ROOT)
print("QC_ROOT:", QC_ROOT)
print("NetCDF backend source:", NETCDF_BACKEND_SOURCE)

MODEL: aurora_cahoy2010_replication_v0
MANIFEST: /home/u11/danielxinhuang/Documents/aurora/roadrunner_egp/aurora_subneptune_grid/manifests/aurora_cahoy2010_replication_v0_manifest.csv
NC_ROOT: /home/u11/danielxinhuang/Documents/aurora/roadrunner_egp/aurora_subneptune_grid/outputs/aurora_cahoy2010_replication_v0/nc
QC_ROOT: /home/u11/danielxinhuang/Documents/aurora/roadrunner_egp/aurora_subneptune_grid/outputs/aurora_cahoy2010_replication_v0/qc_climate
NetCDF backend source: system


In [2]:
manifest = pd.read_csv(MANIFEST)


def resolve_output_nc(output_nc_value: str) -> Path:
    raw = Path(str(output_nc_value))
    if raw.is_absolute() and raw.exists():
        return raw
    p_repo = REPO_ROOT / raw
    if p_repo.exists():
        return p_repo
    p_nc = NC_ROOT / raw.name
    return p_nc


manifest["nc_path"] = manifest["output_nc"].map(resolve_output_nc)
manifest["exists"] = manifest["nc_path"].map(Path.exists)
available = manifest[manifest["exists"]].copy()

if available.empty:
    raise RuntimeError(f"No NetCDF files found for {MODEL} under {NC_ROOT}")

# Climate is phase-independent, so take one representative file per climate group.
# Prefer phase=0 deg when present; otherwise take the nearest available phase.
available["phase_delta_from_zero"] = (available["phase_deg"] - 0.0).abs()
representative = (
    available
    .sort_values(["climate_group_index", "phase_delta_from_zero", "run_index"])
    .groupby("climate_group_index", as_index=False)
    .head(1)
    .sort_values("climate_group_index")
    .reset_index(drop=True)
)

print(f"manifest rows: {len(manifest)}")
print(f"available .nc rows: {len(available)}")
print(f"representative climate groups found: {len(representative)}")

all_groups = sorted(manifest["climate_group_index"].dropna().astype(int).unique())
found_groups = sorted(representative["climate_group_index"].dropna().astype(int).unique())
missing_groups = [g for g in all_groups if g not in found_groups]
print(f"expected climate groups from manifest: {len(all_groups)}")
print(f"missing climate groups (no available representative .nc yet): {missing_groups}")

display(
    representative[
        [
            "climate_group_index",
            "run_index",
            "phase_deg",
            "cahoy_reference_name",
            "nc_path",
        ]
    ]
)

manifest rows: 304
available .nc rows: 261
representative climate groups found: 14
expected climate groups from manifest: 16
missing climate groups (no available representative .nc yet): [np.int64(11), np.int64(15)]


,climate_group_index,run_index,phase_deg,cahoy_reference_name,nc_path
0,0,0,0.0,Jupiter_1x_0.8AU_0deg.dat,/home/u11/danielxinhuang/Documents/aurora/road...
1,1,19,0.0,Jupiter_1x_2AU_0deg.dat,/home/u11/danielxinhuang/Documents/aurora/road...
2,2,38,0.0,Jupiter_1x_5AU_0deg.dat,/home/u11/danielxinhuang/Documents/aurora/road...
3,3,57,0.0,Jupiter_1x_10AU_0deg.dat,/home/u11/danielxinhuang/Documents/aurora/road...
4,4,76,0.0,Jupiter_3x_0.8AU_0deg.dat,/home/u11/danielxinhuang/Documents/aurora/road...
5,5,95,0.0,Jupiter_3x_2AU_0deg.dat,/home/u11/danielxinhuang/Documents/aurora/road...
6,6,114,0.0,Jupiter_3x_5AU_0deg.dat,/home/u11/danielxinhuang/Documents/aurora/road...
7,7,133,0.0,Jupiter_3x_10AU_0deg.dat,/home/u11/danielxinhuang/Documents/aurora/road...
8,8,152,0.0,Neptune_10x_0.8AU_0deg.dat,/home/u11/danielxinhuang/Documents/aurora/road...
9,9,171,0.0,Neptune_10x_2AU_0deg.dat,/home/u11/danielxinhuang/Documents/aurora/road...


In [3]:
qc_rows = []
plot_rows = []

for row in representative.itertuples(index=False):
    nc_path = Path(row.nc_path)
    climate_group = int(row.climate_group_index)

    with xr.open_dataset(nc_path) as ds:
        result = validate_dataset(ds, nc_path)
        qc_row = result_to_row(result, ds)

        qc_row["climate_group_index"] = climate_group
        qc_row["selected_phase_deg"] = float(row.phase_deg)
        qc_row["selected_cahoy_reference_name"] = row.cahoy_reference_name
        qc_rows.append(qc_row)

        run_label = result.run_id or f"run_{int(qc_row.get('run_index') or row.run_index):06d}"
        group_dir = QC_PLOTS / f"climate_group_{climate_group:02d}"

        diagnostic_path = make_qc_plot(ds, result, group_dir / f"{run_label}_diagnostic.png")
        spectrum_path = make_spectrum_plot(ds, result, group_dir / f"{run_label}_spectrum.png")

        plot_rows.append(
            {
                "climate_group_index": climate_group,
                "status": result.status,
                "severity": result.severity,
                "diagnostic_plot": str(diagnostic_path),
                "spectrum_plot": str(spectrum_path),
            }
        )

qc_df = pd.DataFrame(qc_rows).sort_values("climate_group_index").reset_index(drop=True)
plot_df = pd.DataFrame(plot_rows).sort_values("climate_group_index").reset_index(drop=True)

qc_out = QC_ROOT / "qc_climate_summary.csv"
plot_out = QC_ROOT / "qc_climate_plot_index.csv"
qc_df.to_csv(qc_out, index=False)
plot_df.to_csv(plot_out, index=False)

display(
    qc_df[
        [
            "climate_group_index",
            "run_index",
            "status",
            "severity",
            "selected_phase_deg",
            "selected_cahoy_reference_name",
            "max_adiabat_ratio",
            "n_adiabat_violations",
            "max_abs_fnet_irfnet",
            "max_brightness_temperature",
        ]
    ]
)
print(f"wrote {qc_out}")
print(f"wrote {plot_out}")
print(f"plot root: {QC_PLOTS}")

,climate_group_index,run_index,status,severity,selected_phase_deg,selected_cahoy_reference_name,max_adiabat_ratio,n_adiabat_violations,max_abs_fnet_irfnet,max_brightness_temperature
0,0,0,warning,warning,0.0,Jupiter_1x_0.8AU_0deg.dat,1.005130,0,0.902597,
1,1,19,warning,warning,0.0,Jupiter_1x_2AU_0deg.dat,1.006147,0,2.691709,
2,2,38,warning,warning,0.0,Jupiter_1x_5AU_0deg.dat,1.006147,0,3.922012,
3,3,57,warning,warning,0.0,Jupiter_1x_10AU_0deg.dat,1.006147,0,5.499624,
4,4,76,warning,warning,0.0,Jupiter_3x_0.8AU_0deg.dat,1.006674,0,1.097183,
5,5,95,warning,warning,0.0,Jupiter_3x_2AU_0deg.dat,1.006148,0,3.550645,
6,6,114,rerun_recommended,rerun_recommended,0.0,Jupiter_3x_5AU_0deg.dat,32.563700,1,1966.315893,
7,7,133,warning,warning,0.0,Jupiter_3x_10AU_0deg.dat,1.006147,0,8.605168,
8,8,152,warning,warning,0.0,Neptune_10x_0.8AU_0deg.dat,1.007737,0,1.955834,
9,9,171,warning,warning,0.0,Neptune_10x_2AU_0deg.dat,1.006148,0,1.578573,


wrote /home/u11/danielxinhuang/Documents/aurora/roadrunner_egp/aurora_subneptune_grid/outputs/aurora_cahoy2010_replication_v0/qc_climate/qc_climate_summary.csv
wrote /home/u11/danielxinhuang/Documents/aurora/roadrunner_egp/aurora_subneptune_grid/outputs/aurora_cahoy2010_replication_v0/qc_climate/qc_climate_plot_index.csv
plot root: /home/u11/danielxinhuang/Documents/aurora/roadrunner_egp/aurora_subneptune_grid/outputs/aurora_cahoy2010_replication_v0/qc_climate/plots


In [4]:
status_order = ["pass", "warning", "rerun_recommended", "fail"]
status_counts = qc_df["status"].value_counts().reindex(status_order, fill_value=0)

overview_png = QC_ROOT / "qc_climate_overview.png"
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].bar(status_counts.index, status_counts.values, color=["#2ca02c", "#ffbf00", "#ff7f0e", "#d62728"])
axes[0].set_title("Climate-level QC status counts")
axes[0].set_ylabel("N climate groups")
axes[0].tick_params(axis="x", rotation=20)

if "max_abs_fnet_irfnet" in qc_df:
    fnet = pd.to_numeric(qc_df["max_abs_fnet_irfnet"], errors="coerce")
    axes[1].plot(qc_df["climate_group_index"], fnet, marker="o")
    axes[1].set_title("max_abs_fnet_irfnet by climate group")
    axes[1].set_xlabel("climate_group_index")
    axes[1].set_ylabel("max_abs_fnet_irfnet")
    axes[1].grid(alpha=0.3)

if "n_adiabat_violations" in qc_df:
    violations = pd.to_numeric(qc_df["n_adiabat_violations"], errors="coerce").fillna(0)
    axes[2].bar(qc_df["climate_group_index"], violations)
    axes[2].set_title("Adiabat violations by climate group")
    axes[2].set_xlabel("climate_group_index")
    axes[2].set_ylabel("n_adiabat_violations")
    axes[2].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(overview_png, dpi=160, bbox_inches="tight")
plt.show()

print(f"wrote {overview_png}")

wrote /home/u11/danielxinhuang/Documents/aurora/roadrunner_egp/aurora_subneptune_grid/outputs/aurora_cahoy2010_replication_v0/qc_climate/qc_climate_overview.png


In [5]:
def show_climate_group_qc(climate_group_index: int):
    """Show summary row and plot paths for one climate group."""
    row = qc_df[qc_df["climate_group_index"] == climate_group_index]
    if row.empty:
        print(f"No climate group {climate_group_index} in this QC table.")
        return
    display(row)

    matches = plot_df[plot_df["climate_group_index"] == climate_group_index]
    if matches.empty:
        print("No plot index row found.")
        return
    display(matches)


# Example:
# show_climate_group_qc(0)